### Chapter 4.5 - Concise Implementation of Softmax Regression

The concise implementation uses PyTorch layers, loss functions, optimizers, and data loaders while preserving the same training control flow.


In [ ]:
import math
import random
import numpy as np
import torch


### 4.5.1 Defining the Model

#### 1. Intuition

A concise softmax-regression model is a linear layer that maps flattened inputs to class logits.

The model returns logits. The loss function will handle softmax internally.

#### 2. Why this exists

PyTorch layers register parameters automatically, so optimizers can find and update them.


#### 3. Examples

Define a linear classifier for tiny 2 by 2 images and three classes.


In [ ]:
net = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Linear(4, 3),
)
X = torch.randn(2, 1, 2, 2)
logits = net(X)
logits.shape


#### 4. Step-by-step breakdown

`torch.nn.Flatten()` converts each image into a feature vector.

`torch.nn.Linear(4, 3)` maps four pixels to three class logits.

`torch.nn.Sequential` runs the listed layers in order.

The output shape is `(2, 3)`: two examples and three classes.

#### 5. Connection to ML systems

This replaces the manual flattening, weight matrix, bias vector, and logits computation.

#### 6. Common confusion points

- The model returns logits, not probabilities.
- `Sequential` runs layers in the order provided.
- The input size to `Linear` must match flattened feature count.
- Registered parameters can be found with `net.parameters()`.


### 4.5.2 Softmax Revisited

#### 1. Intuition

Softmax is still conceptually part of classification, but PyTorch's `CrossEntropyLoss` combines log-softmax and negative log likelihood internally.

Log-softmax means taking logarithms after softmax in a numerically stable way.

#### 2. Why this exists

Combining these steps avoids numerical problems and reduces unnecessary computation.


#### 3. Examples

Use CrossEntropyLoss directly on logits.


In [ ]:
labels = torch.tensor([0, 2])
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
loss


Softmax is still useful for inspection.


In [ ]:
probs = torch.softmax(logits, dim=1)
probs.sum(dim=1)


#### 4. Step-by-step breakdown

`CrossEntropyLoss` receives raw logits and integer labels.

It internally applies a stable log-softmax and computes the correct-class loss.

`torch.softmax` can be used after the model when you want readable probabilities.

The row sums confirm the inspected probabilities add to 1.

#### 5. Connection to ML systems

In real PyTorch training, pass logits to `CrossEntropyLoss`. Use softmax mainly for interpretation or prediction confidence.

#### 6. Common confusion points

- Do not softmax before `CrossEntropyLoss` in the standard case.
- Labels should be class IDs, not one-hot rows.
- Softmax probabilities are useful for inspection.
- Logits can be any real numbers.


### 4.5.3 Training

#### 1. Intuition

Concise classification training follows: zero gradients, compute logits, compute cross-entropy loss, call backward, step optimizer.

#### 2. Why this exists

The framework shortens the code, but the training order stays the same as the from-scratch version.


#### 3. Examples

One compact training epoch on tiny synthetic image data.


In [ ]:
X = torch.randn(8, 1, 2, 2)
y = torch.tensor([0, 1, 2, 1, 0, 2, 1, 0])
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)
net = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(4, 3))
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.SGD(net.parameters(), lr=0.1)
for Xb, yb in loader:
    opt.zero_grad(); loss_fn(net(Xb), yb).backward(); opt.step()
loss_fn(net(X), y)


#### 4. Step-by-step breakdown

The DataLoader returns batches of images and labels.

The model produces logits.

`CrossEntropyLoss` computes scalar loss from logits and class IDs.

`backward()` computes gradients.

`opt.step()` updates parameters.

#### 5. Connection to ML systems

This is the standard skeleton of many PyTorch classification loops.

#### 6. Common confusion points

- `zero_grad()` must happen before the next backward pass.
- The model should output one logit per class.
- `CrossEntropyLoss` combines softmax-like behavior with loss computation.
- Synthetic data only checks code structure, not real accuracy.


### 4.5.4 Summary

#### 1. Intuition

The concise classifier uses `Flatten`, `Linear`, `CrossEntropyLoss`, `SGD`, and `DataLoader`.

#### 2. Why this exists

These objects package the manual softmax-regression implementation into reusable PyTorch components.


#### 3. Examples

Map concise objects to manual concepts.


In [ ]:
mapping = {
    "Flatten": "reshape images into vectors",
    "Linear": "compute class logits",
    "CrossEntropyLoss": "stable softmax plus loss",
    "SGD": "update parameters",
}
mapping


#### 4. Step-by-step breakdown

Each concise object replaces a manual operation.

The execution order is still the training-loop order.

The mapping is the key to reading concise code without losing the underlying mechanics.

#### 5. Connection to ML systems

Later neural classifiers replace only the model body. The output logits and cross-entropy pattern remain common.

#### 6. Common confusion points

- Concise code should still be explainable line by line.
- The model outputs logits.
- The loss function owns the stable softmax-loss combination.
- Optimizer objects update registered parameters.


### 4.5.5 Exercises

#### 1. Intuition

These exercises practice modifying concise classifier dimensions and reading outputs.

#### 2. Why this exists

Shape control is the main practical skill for concise classification code.


#### 3. Examples

Exercise 1: define a classifier for 4 by 4 grayscale images and five classes.


In [ ]:
net5 = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(16, 5))
X5 = torch.randn(3, 1, 4, 4)
logits5 = net5(X5)
logits5.shape


Exercise 2: compute predictions from logits.


In [ ]:
preds = torch.argmax(logits5, dim=1)
preds.shape


#### 4. Step-by-step breakdown

Exercise 1 checks flattened input size and class count.

Exercise 2 checks prediction shape.

Three examples should produce three predicted class IDs.

#### 5. Connection to ML systems

These checks prevent common mistakes before training on larger image datasets.

#### 6. Common confusion points

- Flattened size is channels times height times width.
- Output size is number of classes.
- Predictions are class IDs.
- Prediction shape should match label shape.
